### IBLM / GAPT — SCAN Length Split Playground

Train HF causal LMs on the SCAN length generalization task with different regularization strategies:
1. **SFT** — pure supervised fine-tuning (no regularization)
2. **SFT + Weight Decay** — SFT with L2 regularization
3. **MBE Regularized** — static phase 2 (compression via Matrix-Based Entropy)
4. **GAPT** — dynamic phase transitions (memorization ↔ compression)

**Usage:** Run cells 1–3 (setup), then pick a config in cell 4 and train.

In [1]:
# Setup
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import random
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from src.gapt_trainer import GaptTrainer, GaptConfig, aggregate_log_history
from train_cpt_scan import (
    load_scan_length_manual, 
    compute_scan_accuracy, 
    check_scan_match,
    ScanAccuracyCallback,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Using device: cpu


In [2]:
# ============================================================
# Load SCAN Dataset (length split)
# ============================================================
SEED = 42
MAX_LENGTH = 128

dataset = load_scan_length_manual()

# Format: "Input: <command>\nOutput: <actions>"
def format_scan(example):
    return {"text": f"Input: {example['commands']}\nOutput: {example['actions']}"}

dataset = dataset.map(format_scan)

# Split: train (short), test_id (held-out short), test_ood (long sequences)
full_train = dataset["train"]
train_test_split = full_train.train_test_split(test_size=0.1, seed=SEED)
train_data = train_test_split["train"]
test_id = train_test_split["test"]
test_ood = dataset["test"]
if len(test_ood) > 500:
    test_ood = test_ood.shuffle(seed=SEED).select(range(500))

print(f"Train: {len(train_data)} | Test ID: {len(test_id)} | Test OOD: {len(test_ood)}")
print(f"Example: {train_data[0]['text'][:120]}")

Map:   0%|          | 0/16990 [00:00<?, ? examples/s]

Map:   0%|          | 0/3920 [00:00<?, ? examples/s]

Train: 15291 | Test ID: 1699 | Test OOD: 500
Example: Input: walk opposite left and turn opposite right thrice
Output: I_TURN_LEFT I_TURN_LEFT I_WALK I_TURN_RIGHT I_TURN_RIGH


In [ ]:
# ============================================================
# Model Init (reusable)
# ============================================================
MODEL_NAME = "Qwen/Qwen3-0.6B"  # Change to test other models

def init_model(model_name=MODEL_NAME):
    """Load a fresh HF causal LM + tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.bfloat16, device_map="auto"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id
    return model, tokenizer

def tokenize_scan(data, tokenizer, max_length=MAX_LENGTH):
    """Tokenize SCAN with answer-only labels (mask question with -100)."""
    def _tokenize(examples):
        texts = examples["text"]
        tokenized = tokenizer(
            texts, padding="max_length", truncation=True, max_length=max_length,
        )
        labels = []
        for i, text in enumerate(texts):
            input_ids = tokenized["input_ids"][i]
            split_marker = "\nOutput: "
            answer_start = text.find(split_marker)
            if answer_start != -1:
                header_text = text[:answer_start + len(split_marker)]
                mask_len = len(tokenizer(header_text, add_special_tokens=False)["input_ids"])
                label = [-100] * mask_len + input_ids[mask_len:]
                label = label[:max_length] + [-100] * max(0, max_length - len(label))
            else:
                label = input_ids.copy()
            labels.append(label)
        tokenized["labels"] = labels
        return tokenized
    
    tok_data = data.map(_tokenize, batched=True, remove_columns=data.column_names)
    tok_data.set_format("torch")
    return tok_data

print(f"Model: {MODEL_NAME}")

---
## Config Selection

Pick one of the 4 configurations below by setting `CONFIG`.

In [ ]:
# ============================================================
# Training Configuration
# ============================================================
CONFIG = "gapt"  # Choose: "sft", "sft_wd", "mbe_regularized", "gapt"

# Shared hyperparams
EPOCHS = 3
LR = 5e-5
BATCH_SIZE = 8
MBE_WEIGHT = 1.0
PATCH_SIZE = 4
MBE_COMP_MODE = "naive"
ACC_EVAL_STEPS = 200
ACC_EVAL_SAMPLES = 200

# Config-specific params
config_map = {
    "sft":              dict(static_phase=True,  initial_phase=1, mbe_weight=0.0, weight_decay=0.0),
    "sft_wd":           dict(static_phase=True,  initial_phase=1, mbe_weight=0.0, weight_decay=0.1),
    "mbe_regularized":  dict(static_phase=True,  initial_phase=2, mbe_weight=MBE_WEIGHT, weight_decay=0.0),
    "gapt":             dict(static_phase=False, initial_phase=1, mbe_weight=MBE_WEIGHT, weight_decay=0.0),
}

cfg = config_map[CONFIG]
print(f"Config: {CONFIG} → {cfg}")

In [ ]:
# ============================================================
# Initialize Model + Tokenize + Build Trainer
# ============================================================
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

model, tokenizer = init_model()

tok_train = tokenize_scan(train_data, tokenizer)
tok_id = tokenize_scan(test_id, tokenizer)
tok_ood = tokenize_scan(test_ood, tokenizer)

gapt_config = GaptConfig(
    patch_size=PATCH_SIZE,
    mode=MBE_COMP_MODE,
    mbe_weight=cfg["mbe_weight"],
    entropy_patience=125,
    mbe_patience=75,
    tau_plateau_m=0.01,
    tau_plateau_a=0.01,
    tau_spike=0.1,
    static_phase=cfg["static_phase"],
    initial_phase=cfg["initial_phase"],
)

acc_callback = ScanAccuracyCallback(
    tokenizer=tokenizer,
    test_id=test_id,
    test_ood=test_ood,
    eval_steps=ACC_EVAL_STEPS,
    max_samples=ACC_EVAL_SAMPLES,
)

output_dir = f"./ckpt/iblm_scan_{CONFIG}"

trainer = GaptTrainer(
    gapt_config=gapt_config,
    model=model,
    train_dataset=tok_train,
    eval_dataset={"id": tok_id, "ood": tok_ood},
    args=TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        logging_steps=20,
        logging_first_step=True,
        eval_strategy="steps",
        eval_steps=100,
        weight_decay=cfg["weight_decay"],
        save_strategy="no",
        eval_on_start=True,
        report_to="none",
        bf16=True,
        dataloader_pin_memory=False,
        seed=SEED,
    ),
    callbacks=[acc_callback],
)

print(f"Trainer ready: {CONFIG} | {len(tok_train)} train samples | model={MODEL_NAME}")

In [ ]:
trainer.train()

In [ ]:
# ============================================================
# Final Accuracy Evaluation
# ============================================================
model.eval()
acc_id = compute_scan_accuracy(model, tokenizer, test_id, max_samples=200)
acc_ood = compute_scan_accuracy(model, tokenizer, test_ood, max_samples=200)
print(f"Config: {CONFIG}")
print(f"  ID Accuracy:  {acc_id:.2%}")
print(f"  OOD Accuracy: {acc_ood:.2%}")

In [ ]:
# ============================================================
# Plot Training Curves
# ============================================================
df = aggregate_log_history(trainer.state.log_history)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f"IBLM/GAPT on SCAN — {CONFIG} ({MODEL_NAME})", fontsize=14)

plots = [
    ("loss", "Total Loss"),
    ("ce_loss", "CE Loss (Memorization)"),
    ("mbe_loss", "MBE Loss (Compression)"),
    ("gapt_phi", "GAPT Phase (1=Mem, 2=Comp)"),
    ("eval_id_acc", "ID Accuracy"),
    ("eval_ood_acc", "OOD Accuracy"),
]

for ax, (key, title) in zip(axes.flat, plots):
    if key in df.columns and df[key].notna().any():
        ax.plot(df["step"], df[key], linewidth=0.8)
        ax.set_title(title)
        ax.set_xlabel("step")
        ax.grid(True, alpha=0.3)
        if "acc" in key:
            ax.set_ylim(-0.05, 1.05)
    else:
        ax.set_title(f"{title} (no data)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Eval Loss Curves (ID vs OOD)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, prefix, title in [
    (axes[0], "eval_id", "ID (in-distribution)"),
    (axes[1], "eval_ood", "OOD (length generalization)"),
]:
    for col, label, color in [
        (f"{prefix}_loss", "Total", "black"),
        (f"{prefix}_ce_loss", "CE", "blue"),
        (f"{prefix}_mbe_loss", "MBE", "red"),
    ]:
        if col in df.columns and df[col].notna().any():
            ax.plot(df["step"], df[col], label=label, color=color, linewidth=0.8)
    ax.set_title(f"{title} Eval Loss")
    ax.set_xlabel("step")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Inspect Predictions (qualitative)
# ============================================================
model.eval()
n_samples = 5

print("=" * 80)
print(f"OOD Predictions ({CONFIG})")
print("=" * 80)

for i in range(min(n_samples, len(test_ood))):
    example = test_ood[i]
    full_text = example["text"]
    input_part = full_text.split("\nOutput:")[0] + "\nOutput:"
    target_action = full_text.split("\nOutput: ")[1].strip()
    
    inputs = tokenizer(input_part, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=128, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    prediction = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    is_correct = check_scan_match(target_action, prediction)
    
    print(f"\n--- Sample {i+1} {'✓' if is_correct else '✗'} ---")
    print(f"Input:    {input_part}")
    print(f"Target:   {target_action}")
    print(f"Predicted:{prediction[:200]}")

model.train()

---
## Compare All Configs

Run the cell below to train all 4 configs sequentially and compare final accuracy.

In [ ]:
# ============================================================
# Run All 4 Configs (sequential comparison)
# ============================================================
results = {}
all_dfs = {}

for config_name, cfg in config_map.items():
    print(f"\n{'='*60}")
    print(f"Training: {config_name}")
    print(f"{'='*60}")
    
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model, tokenizer = init_model()
    tok_train = tokenize_scan(train_data, tokenizer)
    tok_id = tokenize_scan(test_id, tokenizer)
    tok_ood = tokenize_scan(test_ood, tokenizer)
    
    gapt_cfg = GaptConfig(
        patch_size=PATCH_SIZE, mode=MBE_COMP_MODE,
        mbe_weight=cfg["mbe_weight"],
        entropy_patience=125, mbe_patience=75,
        tau_plateau_m=0.01, tau_plateau_a=0.01, tau_spike=0.1,
        static_phase=cfg["static_phase"],
        initial_phase=cfg["initial_phase"],
    )
    
    acc_cb = ScanAccuracyCallback(
        tokenizer=tokenizer, test_id=test_id, test_ood=test_ood,
        eval_steps=ACC_EVAL_STEPS, max_samples=ACC_EVAL_SAMPLES,
    )
    
    t = GaptTrainer(
        gapt_config=gapt_cfg, model=model,
        train_dataset=tok_train,
        eval_dataset={"id": tok_id, "ood": tok_ood},
        args=TrainingArguments(
            output_dir=f"./ckpt/iblm_scan_{config_name}",
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            num_train_epochs=EPOCHS, learning_rate=LR,
            logging_steps=20, logging_first_step=True,
            eval_strategy="steps", eval_steps=100,
            weight_decay=cfg["weight_decay"],
            save_strategy="no", eval_on_start=True,
            report_to="none", bf16=True,
            dataloader_pin_memory=False, seed=SEED,
        ),
        callbacks=[acc_cb],
    )
    t.train()
    
    model.eval()
    acc_id = compute_scan_accuracy(model, tokenizer, test_id, 200)
    acc_ood = compute_scan_accuracy(model, tokenizer, test_ood, 200)
    results[config_name] = {"id_acc": acc_id, "ood_acc": acc_ood}
    all_dfs[config_name] = aggregate_log_history(t.state.log_history)
    
    del model, tokenizer, t
    torch.cuda.empty_cache()
    print(f"  → ID: {acc_id:.2%} | OOD: {acc_ood:.2%}")

print(f"\n{'='*60}")
print("Final Results:")
print(f"{'='*60}")
for name, r in results.items():
    print(f"  {name:20s}  ID={r['id_acc']:.2%}  OOD={r['ood_acc']:.2%}")

In [ ]:
# ============================================================
# Compare Loss Curves Across Configs
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"IBLM/GAPT Config Comparison — SCAN ({MODEL_NAME})", fontsize=14)

colors = {"sft": "gray", "sft_wd": "blue", "mbe_regularized": "orange", "gapt": "red"}

for key, title, ax in [
    ("ce_loss", "CE Loss", axes[0]),
    ("mbe_loss", "MBE Loss", axes[1]),
    ("loss", "Total Loss", axes[2]),
]:
    for name, df in all_dfs.items():
        if key in df.columns and df[key].notna().any():
            ax.plot(df["step"], df[key], label=name, color=colors.get(name, "black"), linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("step")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Bar chart of final accuracy
fig, ax = plt.subplots(figsize=(8, 5))
names = list(results.keys())
id_accs = [results[n]["id_acc"] for n in names]
ood_accs = [results[n]["ood_acc"] for n in names]
x = np.arange(len(names))
ax.bar(x - 0.2, id_accs, 0.35, label="ID", color="steelblue")
ax.bar(x + 0.2, ood_accs, 0.35, label="OOD", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title(f"SCAN Accuracy — {MODEL_NAME}")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()